In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ==========================================
# 1. Core Transformer Architecture (Bidirectional)
# ==========================================
class BidirectionalTransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        # self.self_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.norm1 = nn.LayerNorm(d_model)
        
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # Full bidirectional attention (no causal mask)
        # attn_out, _ = self.self_attn(x, x, x)
        batch_size, seq_len, _ = x.size()
        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        # This automatically triggers FlashAttention-2 on H100 GPUs!
        attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=False)

        # Reshape back to original
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        attn_out = self.out_proj(attn_out)
        x = self.norm1(x + attn_out)
        mlp_out = self.mlp(x)
        x = self.norm2(x + mlp_out)
        return x

In [3]:
# ==========================================
# 2. TRM Diffusion Model Setup
# ==========================================
class TRM_Diffusion(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_heads=8, d_ff=2048, num_layers=2, mask_token_id=-100):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.mask_token_id = mask_token_id
        
        # Embeddings
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.mask_emb = nn.Parameter(torch.randn(1, 1, d_model)) # Special embedding for MASK
        self.pos_emb = nn.Embedding(4096, d_model) # Assuming max sequence length 4096
        
        # TRM "Tiny Network" Backbone
        self.layers = nn.ModuleList([
            BidirectionalTransformerLayer(d_model, n_heads, d_ff) 
            for _ in range(num_layers)
        ])
        
        # Output head to predict vocabulary
        self.output_head = nn.Linear(d_model, vocab_size)
        
        # Initialization for the latent space z
        self.z_init = nn.Parameter(torch.randn(1, 1, d_model))

        self.q_head = nn.Sequential(
            nn.Linear(self.d_model, self.d_model // 2),
            nn.ReLU(),
            nn.Linear(self.d_model // 2, 1),
            # nn.Sigmoid() 
        )

    def get_embeddings(self, tokens):
            # Handle regular tokens and the special MASK token
            seq_len = tokens.size(1)
            positions = torch.arange(seq_len, device=tokens.device).unsqueeze(0).expand_as(tokens)
            
            emb = self.token_emb(torch.clamp(tokens, min=0)) # Clamp prevents error on mask_token_id
            
            # Apply special mask embedding where necessary
            mask_positions = (tokens == self.mask_token_id).unsqueeze(-1)
            emb = torch.where(mask_positions, self.mask_emb, emb)
            
            return emb + self.pos_emb(positions)

    def latent_recursion(self, x_t_emb, z, n ):
            """x_t_emb is the combined prompt + masked response embeddings"""
            for _ in range(n):
                # We now add the sequence to the latent state
                combined_state = x_t_emb + z 
                
                for layer in self.layers:
                    combined_state = layer(combined_state)
                
                z = combined_state
                
            logits = self.output_head(z)
            return logits, z

    def deep_recursion(self, x_t_emb, z, n=6, T=3):
        """
        The original TRM Deep Recursion block. 
        T=3 means it does 2 'thought' loops without gradients, 
        and 1 'action' loop with gradients.
        """
        # 1. The "Thinking" Phase (Saves massive VRAM)
        with torch.no_grad():
            for _ in range(T - 1):
                # We don't care about the logits here, just evolving z
                _, z = self.latent_recursion(x_t_emb, z, n)
                
        # 2. The "Action" Phase (Gradients tracked!)
        logits, z = self.latent_recursion(x_t_emb, z, n)
        
        # 3. Get the Confidence Score (The Halting Head)
        q_hat = self.get_q_hat(z)
        
        # 4. Markovian Detachment to sever the graph for the next N_SUP step
        return logits, z.detach(), q_hat
    
    def get_q_hat(self, z):
        """
        Compresses the latent state and returns a confidence score (0.0 to 1.0)
        indicating if the model believes it has solved the masked sequence.
        """
        z_mean = z.mean(dim=1) 
        return self.q_head(z_mean).squeeze(-1) # Returns shape (batch_size,)

Train the tokenizer

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import os

# ==========================================
# 1. Train a Custom Tokenizer on TinyStories
# ==========================================
def train_custom_tokenizer(dataset, vocab_size=10000, save_path="tinystories_tokenizer.json"):
    """
    Trains a Byte-Pair Encoding (BPE) tokenizer from scratch.
    """
    if os.path.exists(save_path):
        print(f"Loading existing tokenizer from {save_path}...")
        return Tokenizer.from_file(save_path)
        
    print("Training new tokenizer from scratch. This might take a minute...")
    
    # Initialize a BPE Tokenizer
    tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = Whitespace()

    # Define our special tokens
    # [PAD] = 0, [UNK] = 1, [BOS] = 2, [EOS] = 3
    special_tokens = ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
    
    # Configure the trainer
    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=special_tokens,
        show_progress=True
    )

    # We use a generator to avoid loading the entire dataset into RAM
    def batch_iterator(batch_size=1000):
        for i in range(0, len(dataset), batch_size):
            yield dataset[i : i + batch_size]["text"]

    # Train and save
    tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)
    tokenizer.save(save_path)
    print("Tokenizer training complete!")
    
    return tokenizer

# ==========================================
# 2. PyTorch Dataset for TRM-Diffusion
# ==========================================
class TinyStoriesDiffusionDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_length=256):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        # Grab special token IDs
        self.pad_id = self.tokenizer.token_to_id("[PAD]")
        self.bos_id = self.tokenizer.token_to_id("[BOS]")
        self.eos_id = self.tokenizer.token_to_id("[EOS]")

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # 1. Get text and encode
        text = self.dataset[idx]['text']
        encoded = self.tokenizer.encode(text).ids
        
        # 2. Add [EOS] to the end
        encoded = encoded + [self.eos_id]
        
        # 3. Truncate or Pad
        if len(encoded) > self.max_length:
            # Truncate if too long (keep EOS at the end)
            encoded = encoded[:self.max_length - 1] + [self.eos_id]
        else:
            # Pad with EOS tokens if too short 
            # (LLaDA strategy: pad with EOS so it learns to predict them!)
            padding_length = self.max_length - len(encoded)
            encoded = encoded + ([self.eos_id] * padding_length)
            
        # Suppose we want the first 5 tokens to act as the prompt
        split_idx = 5

        # The prompt is the beginning of the story
        prompt_tokens = [self.bos_id] + encoded[:split_idx]

        # The target is the rest of the story
        target_tokens = encoded[split_idx:]

        prompt = torch.tensor(prompt_tokens, dtype=torch.long)
        target = torch.tensor(target_tokens, dtype=torch.long)
        
        return prompt, target

# ==========================================
# 3. Putting it together: The Setup Function
# ==========================================
def get_dataloaders(batch_size=32, max_length=256, vocab_size=10000):
    print("Downloading/Loading TinyStories...")
    # Load the training and validation splits
    ds_train = load_dataset("roneneldan/TinyStories", split="train")
    ds_val = load_dataset("roneneldan/TinyStories", split="validation")
    
    # Train/Load Tokenizer
    tokenizer = train_custom_tokenizer(ds_train, vocab_size=vocab_size)
    
    # Create PyTorch Datasets
    train_dataset = TinyStoriesDiffusionDataset(ds_train, tokenizer, max_length=max_length)
    val_dataset = TinyStoriesDiffusionDataset(ds_val, tokenizer, max_length=max_length)
    
    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=True)
    
    return train_loader, val_loader, tokenizer

# ==========================================
# Usage Example
# ==========================================
if __name__ == "__main__":
    # Hyperparameters
    BATCH_SIZE = 16
    MAX_SEQ_LENGTH = 256
    VOCAB_SIZE = 10000  # Smaller vocab size is great for tiny models!
    
    train_loader, val_loader, tokenizer = get_dataloaders(
        batch_size=BATCH_SIZE, 
        max_length=MAX_SEQ_LENGTH, 
        vocab_size=VOCAB_SIZE
    )
    
    # Test a single batch
    for prompts, targets in train_loader:
        print(f"Prompt shape (BOS token): {prompts.shape}")
        print(f"Targets shape (Story): {targets.shape}")
        
        # Decode the first sequence in the batch just to verify
        print("\nDecoded target 0:")
        # We replace EOS id with string just to visualize the padding
        decoded_words = [tokenizer.id_to_token(id) for id in targets[0].tolist()]
        print(" ".join(decoded_words[:50]) + " ...")
        break

c:\Users\MSI-1\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Downloading/Loading TinyStories...
Training new tokenizer from scratch. This might take a minute...
Tokenizer training complete!
Prompt shape (BOS token): torch.Size([16, 1])
Targets shape (Story): torch.Size([16, 256])

Decoded target 0:
Once there was a young girl called Lucy . One day , Lucy went to the park with her mom . When she arrived , she saw a big , long balloon . It was bright and colourful and she wanted to have it . She ran to it and ...


In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import os
# ==========================================
# 2. PyTorch Dataset for TRM-Diffusion
# ==========================================
class TinyStoriesDiffusionDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_length=256):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        # Grab special token IDs
        self.pad_id = self.tokenizer.token_to_id("[PAD]")
        self.bos_id = self.tokenizer.token_to_id("[BOS]")
        self.eos_id = self.tokenizer.token_to_id("[EOS]")

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # 1. Get text and encode
        text = self.dataset[idx]['text']
        encoded = self.tokenizer.encode(text).ids
        
        # 2. Add [EOS] to the end
        encoded = encoded + [self.eos_id]
        
        # 3. Truncate or Pad
        if len(encoded) > self.max_length:
            # Truncate if too long (keep EOS at the end)
            encoded = encoded[:self.max_length - 1] + [self.eos_id]
        else:
            # Pad with EOS tokens if too short 
            # (LLaDA strategy: pad with EOS so it learns to predict them!)
            padding_length = self.max_length - len(encoded)
            encoded = encoded + ([self.eos_id] * padding_length)
            
        # Suppose we want the first 5 tokens to act as the prompt
        split_idx = 5

        # The prompt is the beginning of the story
        prompt_tokens = [self.bos_id] + encoded[:split_idx]

        # The target is the rest of the story
        target_tokens = encoded[split_idx:]

        prompt = torch.tensor(prompt_tokens, dtype=torch.long)
        target = torch.tensor(target_tokens, dtype=torch.long)
        
        return prompt, target

# ==========================================
# 3. Putting it together: The Setup Function
# ==========================================
def get_dataloaders(batch_size=32, max_length=256, vocab_size=10000):
    print("Downloading/Loading TinyStories...")
    # Load the training and validation splits
    ds_train = load_dataset("roneneldan/TinyStories", split="train[:100000]")
    ds_val = load_dataset("roneneldan/TinyStories", split="validation")
    
    # Train/Load Tokenizer
    tokenizer = Tokenizer.from_file("/workspace/G-TRM-NAR-/tinystories_tokenizer.json")
    
    # Create PyTorch Datasets
    train_dataset = TinyStoriesDiffusionDataset(ds_train, tokenizer, max_length=max_length)
    val_dataset = TinyStoriesDiffusionDataset(ds_val, tokenizer, max_length=max_length)
    
    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=True)
    
    return train_loader, val_loader, tokenizer



l40s

In [5]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.amp import GradScaler, autocast
from tqdm import tqdm
import wandb

def train_l40s_original_trm():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device} | L40S + Original TRM Algorithm")

    # ==========================================
    # 1. L40S 48GB Hyperparameters
    # ==========================================
    BATCH_SIZE = 64             # Massive true batch size! No accumulation needed.
    MAX_SEQ_LENGTH = 512        # Optimized for TinyStories
    VOCAB_SIZE = 10000
    MASK_TOKEN_ID = -100
    
    D_MODEL = 512             
    N_HEADS = 8               
    D_FF = 2048                
    NUM_LAYERS = 2            
    
    N_SUP = 16                 
    N_RECURSIONS = 6          
    EPOCHS = 10  
    LEARNING_RATE = 4e-4

    wandb.init(project="TRM-Diffusion", name="L40S-Original-TRM", config=locals())

    # ==========================================
    # 2. Setup Data and Model
    # ==========================================
    # (Assuming Subset logic is added in get_dataloaders to cap at 1M stories)
    train_loader, val_loader, tokenizer = get_dataloaders(
        batch_size=BATCH_SIZE, max_length=MAX_SEQ_LENGTH, vocab_size=VOCAB_SIZE
    )

    model = TRM_Diffusion(
        vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_heads=N_HEADS,
        d_ff=D_FF, num_layers=NUM_LAYERS, mask_token_id=MASK_TOKEN_ID
    ).to(device)

    model = torch.compile(model, mode="max-autotune")
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.1)
    scaler = GradScaler(device='cuda')

    # ==========================================
    # 3. The Original TRM Training Loop
    # ==========================================
    for epoch in range(EPOCHS):
        model.train()
        total_epoch_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        
        for batch_idx, (_, targets) in enumerate(pbar):
            targets = targets.to(device)
            batch_size, seq_len = targets.size()
            
            prompt_length = 3 
            t = max(torch.rand(1).item(), 0.05)
            
            mask_probs = torch.zeros((batch_size, seq_len), device=device)
            mask_probs[:, prompt_length:] = t  
            mask_boolean = torch.bernoulli(mask_probs).bool()
            
            if mask_boolean.sum() == 0: continue

            x_t = targets.clone()
            x_t[mask_boolean] = MASK_TOKEN_ID
            
            z = model.z_init.expand(batch_size, seq_len, model.d_model)
            
            # For tracking metrics across the unrolled steps
            batch_text_loss = 0.0
            steps_taken = 0
            
            # --- THE TRM DEEP SUPERVISION LOOP ---
            for step in range(N_SUP):
                steps_taken += 1
                optimizer.zero_grad()
                
                with autocast(device_type='cuda', dtype=torch.bfloat16):
                    # 1. Re-embed x_t
                    x_t_emb = model.get_embeddings(x_t)
                    
                    # 2. ONE LINE DOING ALL THE HEAVY LIFTING!
                    # This handles the T-loops, the n-recursions, the detachment, and the Q-score.
                    logits, z, q_hat = model.deep_recursion(x_t_emb, z, n=N_RECURSIONS, T=3)
                    
                    # 3. Text Loss
                    logits_flat = logits.view(-1, VOCAB_SIZE)
                    targets_flat = targets.view(-1)
                    mask_flat = mask_boolean.view(-1)
                    
                    step_text_loss = F.cross_entropy(logits_flat[mask_flat], targets_flat[mask_flat])
                    scaled_text_loss = step_text_loss / t
                    
                    # 4. Q-Head Ground Truth (Accuracy)
                    with torch.no_grad():
                        preds = logits.argmax(dim=-1)
                        correct_mask = (preds == targets) & mask_boolean
                        tokens_masked = mask_boolean.sum(dim=1).clamp(min=1)
                        y_true_accuracy = correct_mask.sum(dim=1).float() / tokens_masked
                    
                    q_loss = F.binary_cross_entropy_with_logits(q_hat, y_true_accuracy)
                    
                    # Total loss for this step
                    total_step_loss = scaled_text_loss + q_loss
                
                # 5. INSTANT BACKPROPAGATION
                scaler.scale(total_step_loss).backward()
                scaler.step(optimizer)
                scaler.update()
                
                batch_text_loss += scaled_text_loss.item()
                
                # 6. EARLY STOPPING
                if (q_hat > 0.95).all():
                    break
                    
            # --- END OF N_SUP LOOP ---
            
            avg_batch_loss = batch_text_loss / steps_taken
            total_epoch_loss += avg_batch_loss
            
            wandb.log({
                "train/text_loss": avg_batch_loss,
                "train/q_loss": q_loss.item(),
                "train/steps_taken": steps_taken,
                "train/avg_batch_accuracy": y_true_accuracy.mean().item(),
            })
            
            pbar.set_postfix({
                "Loss": f"{(total_epoch_loss / (batch_idx + 1)):.4f}",
                "Steps": steps_taken
            })

        print(f"Epoch {epoch+1} Avg Loss: {total_epoch_loss / len(train_loader):.4f}")
        torch.save(model.state_dict(), f"trm_l40s_original_epoch_{epoch+1}.pt")
    
    wandb.finish()

if __name__ == "__main__":
    train_l40s_original_trm()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Training on: cuda | L40S + Original TRM Algorithm


wandb: Currently logged in as: rao-adnan-098 (rao-adnan-098-jamia-millia-islamia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Downloading/Loading TinyStories...


Epoch 1/10: 100%|██████████| 1562/1562 [1:25:26<00:00,  3.28s/it, Loss=12.1850, Steps=16]


...

Epoch 4 Avg Loss: 11.1506


Epoch 5/10:   2%|▏         | 27/1562 [01:29<1:24:38,  3.31s/it, Loss=12.0733, Steps=16]


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f912a1cc8c0>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f932644cb60, execution_count=5 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7f9326edfe30, raw_cell="import torch
import torch.nn.functional as F
from .." transformed_cell="import torch
import torch.nn.functional as F
from .." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Brunpod_1/workspace/G-TRM-NAR-/ddiffusion.ipynb#X10sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

another epoch

In [ ]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.amp import GradScaler, autocast
from tqdm import tqdm
import wandb

# (Assume TRM_Diffusion and get_dataloaders are imported from your main file)

def resume_l40s_epoch3():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Resuming Training on: {device} | L40S + Softened Halting")

    # ==========================================
    # 1. Hyperparameters (Must match Epoch 1 & 2 perfectly)
    # ==========================================
    BATCH_SIZE = 64             
    MAX_SEQ_LENGTH = 512        
    VOCAB_SIZE = 10000
    MASK_TOKEN_ID = -100
    
    D_MODEL = 512             
    N_HEADS = 8               
    D_FF = 2048                
    NUM_LAYERS = 2            
    
    N_SUP = 16                 
    N_RECURSIONS = 6          
    LEARNING_RATE = 4e-4

    # Tracking as a new run so it doesn't overwrite your Epoch 1-2 charts
    wandb.init(project="TRM-Diffusion", name="L40S-Epoch-3-Resume", config=locals())

    # ==========================================
    # 2. Setup Data and Model
    # ==========================================
    train_loader, val_loader, tokenizer = get_dataloaders(
        batch_size=BATCH_SIZE, max_length=MAX_SEQ_LENGTH, vocab_size=VOCAB_SIZE
    )

    model = TRM_Diffusion(
        vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_heads=N_HEADS,
        d_ff=D_FF, num_layers=NUM_LAYERS, mask_token_id=MASK_TOKEN_ID
    ).to(device)

    # --- THE RESUME LOGIC ---
    checkpoint_path = "trm_l40s_original_epoch_2.pt"
    print(f"Loading weights from: {checkpoint_path}")
    model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))

    print("Compiling model...")
    model = torch.compile(model, mode="max-autotune")
    
    # Reinitialize optimizer (momentum will rebuild quickly in the first few batches)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.1)
    scaler = GradScaler(device='cuda')

    # ==========================================
    # 3. The Epoch 3 Training Loop
    # ==========================================
    # We set the range to specifically run only Epoch 3
    for epoch in range(2, 3): 
        model.train()
        total_epoch_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} (Resumed)")
        
        for batch_idx, (_, targets) in enumerate(pbar):
            targets = targets.to(device)
            batch_size, seq_len = targets.size()
            
            prompt_length = 3 
            t = max(torch.rand(1).item(), 0.05)
            
            mask_probs = torch.zeros((batch_size, seq_len), device=device)
            mask_probs[:, prompt_length:] = t  
            mask_boolean = torch.bernoulli(mask_probs).bool()
            
            if mask_boolean.sum() == 0: continue

            x_t = targets.clone()
            x_t[mask_boolean] = MASK_TOKEN_ID
            
            z = model.z_init.expand(batch_size, seq_len, model.d_model)
            
            batch_text_loss = 0.0
            steps_taken = 0
            
            # --- THE TRM DEEP SUPERVISION LOOP ---
            for step in range(N_SUP):
                steps_taken += 1
                optimizer.zero_grad()
                
                with autocast(device_type='cuda', dtype=torch.bfloat16):
                    x_t_emb = model.get_embeddings(x_t)
                    logits, z, q_hat = model.deep_recursion(x_t_emb, z, n=N_RECURSIONS)
                    
                    logits_flat = logits.view(-1, VOCAB_SIZE)
                    targets_flat = targets.view(-1)
                    mask_flat = mask_boolean.view(-1)
                    
                    step_text_loss = F.cross_entropy(logits_flat[mask_flat], targets_flat[mask_flat])
                    scaled_text_loss = step_text_loss / t
                    
                    with torch.no_grad():
                        preds = logits.argmax(dim=-1)
                        correct_mask = (preds == targets) & mask_boolean
                        tokens_masked = mask_boolean.sum(dim=1).clamp(min=1)
                        y_true_accuracy = correct_mask.sum(dim=1).float() / tokens_masked
                    
                    q_loss = F.binary_cross_entropy_with_logits(q_hat, y_true_accuracy)
                    total_step_loss = scaled_text_loss + q_loss
                
                scaler.scale(total_step_loss).backward()
                scaler.step(optimizer)
                scaler.update()
                
                z = z.detach()
                batch_text_loss += scaled_text_loss.item()
                
                # --- THE SPEEDUP: SOFTENED HALTING ---
                q_hat_prob = torch.sigmoid(q_hat)
                # Instead of .all(), we check if the average confidence across the batch is > 90%
                if q_hat_prob.mean() > 0.90:
                    break
                    
            avg_batch_loss = batch_text_loss / steps_taken
            total_epoch_loss += avg_batch_loss
            
            wandb.log({
                "train/text_loss": avg_batch_loss,
                "train/q_loss": q_loss.item(),
                "train/steps_taken": steps_taken,
                "train/avg_batch_accuracy": y_true_accuracy.mean().item(),
            })
            
            pbar.set_postfix({
                "Loss": f"{(total_epoch_loss / (batch_idx + 1)):.4f}",
                "Steps": steps_taken
            })

        print(f"Epoch {epoch+1} Avg Loss: {total_epoch_loss / len(train_loader):.4f}")
        torch.save(model.state_dict(), f"trm_l40s_original_epoch_{epoch+1}.pt")
    
    wandb.finish()

if __name__ == "__main__":
    resume_l40s_epoch3()

In [17]:
import torch
import torch.nn.functional as F

# (Assume TRM_Diffusion and get_dataloaders/tokenizer are imported)

@torch.no_grad()
def generate_conditional(model, tokenizer, prompt_text, max_seq_len=512, sampling_steps=128):
    """
    Generates text using the reverse diffusion process.
    Optimized for L40S with Original TRM Q-Head Dynamic Halting!
    """
    device = next(model.parameters()).device
    model.eval()

    # 1. Setup the initial sequence (Prompt + Masks)
    prompt_ids = tokenizer.encode(prompt_text).ids
    prompt_length = len(prompt_ids)
    answer_length = max_seq_len - prompt_length
    
    mask_token_id = model.mask_token_id
    mask_padding = [mask_token_id] * answer_length
    
    # x_t shape: (1, max_seq_len)
    x_t = torch.tensor([prompt_ids + mask_padding], dtype=torch.long, device=device)
    r_t = x_t.clone()

    N_SUP = 16
    N_RECURSIONS = 6
    
    # Timesteps from 1.0 down to 0.0
    timesteps = torch.linspace(1.0, 1.0 / sampling_steps, sampling_steps, device=device)

    print(f"\nGenerating story from prompt: '{prompt_text}'")
    print(f"Running {sampling_steps} diffusion steps...")

    # Autocast for L40S BFloat16 speed
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        for t_idx, t in enumerate(timesteps):
            # Calculate next timestep 's'
            s = t - (1.0 / sampling_steps) if t_idx < len(timesteps) - 1 else torch.tensor(0.0)
            
            # 1. Embed current sequence and initialize latent z
            r_t_emb = model.get_embeddings(r_t)
            z = model.z_init.expand(1, max_seq_len, model.d_model)
            
            # 2. Dynamic TRM Reasoning
            for step in range(N_SUP):
                # Unpacking all 3 values (Logits, Detached Z, and Q-Head Confidence)
                logits, z, q_hat = model.deep_recursion(r_t_emb, z, n=N_RECURSIONS)
                
                # SPEED OPTIMIZATION: Early Stopping via Q-Head!
                # If the model is >95% confident it has solved this step, stop thinking.
                # q_hat_prob = torch.sigmoid(q_hat)
                # if (q_hat_prob > 0.95).all():
                #     break
            
            # 3. Predict and Calculate True Confidence
            vocab_dim = logits.size(-1)
            eos_id = tokenizer.token_to_id("[EOS]") 
            pad_id = tokenizer.token_to_id("[PAD]") 
            
            pure_probs = F.softmax(logits, dim=-1)
            confidences, _ = torch.max(pure_probs, dim=-1)
            
            top_k = 50
            temperature = 0.8
            
            # Hard ban [PAD] everywhere
            logits[:, :, pad_id] = -float('inf')
            
            # --- THE FIX: SPATIAL EOS BANNING ---
            # Force the model to write AT LEAST 40 words before it is allowed to stop.
            # This prevents the "empty string" bug, but prevents the "500-word salad" bug!
            min_story_length = 40
            safe_length = min(prompt_length + min_story_length, max_seq_len)
            logits[:, :safe_length, eos_id] = -float('inf')
            
            # Isolate Top 50 best guesses
            top_k_logits, top_k_indices = torch.topk(logits, top_k, dim=-1)
            filtered_logits = torch.full_like(logits, -float('inf'))
            filtered_logits.scatter_(2, top_k_indices, top_k_logits)
            
            # Sample with temperature
            filtered_probs = F.softmax(filtered_logits / temperature, dim=-1)
            flat_probs = filtered_probs.view(-1, vocab_dim) 
            sampled_tokens = torch.multinomial(flat_probs, num_samples=1)
            predicted_tokens = sampled_tokens.view(1, max_seq_len)
            
            # 4. Update the sequence
            is_masked = (r_t == mask_token_id)
            r_0_hat = torch.where(is_masked, predicted_tokens, r_t)
            
            # 5. Low-Confidence Remasking (LLaDA Logic)
            num_to_remask = int(answer_length * s.item())
            
            if num_to_remask > 0:
                # Protect the prompt from being masked
                confidences[:, :prompt_length] = float('inf')
                
                # Protect [EOS] tokens from being re-masked if the model naturally placed them!
                confidences[r_0_hat == eos_id] = float('inf')
                
                # Find the least confident words and mask them again
                _, indices_to_remask = torch.topk(confidences, num_to_remask, dim=1, largest=False)
                
                r_t = r_0_hat.clone()
                r_t.scatter_(1, indices_to_remask, mask_token_id)
            else:
                r_t = r_0_hat

    # 6. Clean up the output 
    generated_ids = r_t[0].tolist()
    
    # Debug print so we can see what it generated before truncation!
    print(f"\n[DEBUG] Raw Token IDs (First 20): {generated_ids[:20]}")
    
    if eos_id in generated_ids:
        generated_ids = generated_ids[:generated_ids.index(eos_id)]
        
    raw_text = tokenizer.decode(generated_ids)
    return raw_text.replace("[PAD]", "").strip()


In [18]:
# ==========================================
# Evaluation Wrapper
# ==========================================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # CRITICAL FIX: Match the L40S Training Hyperparameters!
    MAX_SEQ_LENGTH = 512       
    VOCAB_SIZE = 10000
    MASK_TOKEN_ID = -100
    
    _, _, tokenizer = get_dataloaders(batch_size=1, max_length=MAX_SEQ_LENGTH, vocab_size=VOCAB_SIZE)
    
    model = TRM_Diffusion(
        vocab_size=VOCAB_SIZE,
        d_model=512,
        n_heads=8,
        d_ff=2048,
        num_layers=2,
        mask_token_id=MASK_TOKEN_ID
    ).to(device)
    
    # 1. Load the raw dictionary from the file
    checkpoint_path = "trm_l40s_original_epoch_4.pt" 
    print(f"Loading checkpoint: {checkpoint_path}")
    raw_state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
    
    # 2. Strip the "_orig_mod." prefix from all the keys
    clean_state_dict = {}
    for key, value in raw_state_dict.items():
        # If the key starts with _orig_mod., remove it
        if key.startswith("_orig_mod."):
            clean_key = key.replace("_orig_mod.", "")
            clean_state_dict[clean_key] = value
        else:
            clean_state_dict[key] = value
            
    # 3. Load the cleaned dictionary into your model
    model.load_state_dict(clean_state_dict)
    
    # 4. NOW you can compile it for inference/resuming
    print("Compiling model...")
    model = torch.compile(model, mode="max-autotune")
    
    test_prompts = [
        "Once upon a time,",
        "The angry bear",
        "Lily found a magic"
    ]
    
    print("\n" + "="*50)
    for prompt in test_prompts:
        story = generate_conditional(
            model, 
            tokenizer, 
            prompt, 
            max_seq_len=MAX_SEQ_LENGTH, 
            sampling_steps=128
        )
        print(f"\nPROMPT: {prompt}")
        print(f"STORY: {story}")
        print("-" * 50)

Downloading/Loading TinyStories...
Loading checkpoint: trm_l40s_original_epoch_4.pt
Compiling model...


Generating story from prompt: 'Once upon a time,'
Running 128 diffusion steps...

[DEBUG] Raw Token IDs (First 20): [228, 178, 17, 17, 324, 17, 187, 17, 2908, 379, 179, 7626, 291, 178, 17, 17, 306, 178, 17, 379]

PROMPT: Once upon a time,
STORY: she and . . girl . was . oc loved to concent happy and . . One and . loved and every and shell . deed and . she armor Squirrel they . wiggling happy . shell of She mustaches and . ' recording irst
--------------------------------------------------

Generating story from prompt: 'The angry bear'
Running 128 diffusion steps...

[DEBUG] Raw Token IDs (First 20): [739, 15, 17, 285, 17, 378, 5702, 17, 17, 379, 179, 258, 178, 6664, 6393, 324, 6393, 17, 17, 5370]

PROMPT: The angry bear
STORY: As , . little . named particular . . loved to play and Tam Squirrel girl Squirrel . . flat accident . . . a the concent she . . . ! recording and ! . created